# 04 · Fitting the pattern × space-time-correlation cut

            Replaces `3D-cut.ipynb` + `3D-cut_multi_process.ipynb`.

            Inputs: a list of ``de_result.npz`` files produced by
            ``scripts/run_de_sim.py``.

            Outputs: ``cut_config/k_st_coefficients.npz`` and
            ``cut_config/b_coefficients.npz`` (legacy schema preserved).

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
from pathlib import Path
            from relics_de_sim.cuts import (
                PatternSTCut,
                fit_pattern_st_cut,
                pattern_likelihood,
            )
            from relics_de_sim.io import load_de_result

## 1 – Load the DE batch outputs

In [ ]:
de_files = sorted(Path('outputs/de_sim').glob('batch_*/de_result.npz'))
            assert de_files, 'No DE outputs found - run scripts/run_de_sim.py first.'
            print(f'aggregating {len(de_files)} files')

            pattern, st_cor, area = [], [], []
            for fp in de_files:
                d = load_de_result(fp)
                pattern.append(d['pile_up_pattern_coef'])
                st_cor.append(np.log(np.maximum(d['pile_up_st_cor'], 1e-300)))
                area.append(d['pile_up_area'])
            pattern = np.concatenate(pattern); st_cor = np.concatenate(st_cor); area = np.concatenate(area)
            print(f'{len(area):_} pile-up events total')

## 2 – 2-D distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
            h = ax.hist2d(st_cor, pattern, bins=80, cmap='viridis')
            ax.set_xlabel('log(st_cor)'); ax.set_ylabel('pattern likelihood')
            plt.colorbar(h[3], ax=ax)
            plt.show()

## 3 – Fit the area-dependent linear cut

In [ ]:
cut, diag = fit_pattern_st_cut(area, pattern, st_cor, quantile=0.99)
            print(cut)
            cut.to_npz('cut_config/k_st_coefficients.npz', 'cut_config/b_coefficients.npz')

## 4 – Visualise the fit

In [ ]:
bins = diag['area_bins']
            centres = diag['bin_centres']
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            axes[0].plot(centres, diag['slopes'], 'o', label='per-bin slope')
            axes[0].plot(centres, cut.slope(centres), label='global fit')
            axes[0].set_xlabel('area'); axes[0].set_ylabel('slope k_st(area)'); axes[0].legend()
            axes[1].plot(centres, diag['intercepts'], 'o', label='per-bin intercept')
            axes[1].plot(centres, cut.intercept(centres), label='global fit')
            axes[1].set_xlabel('area'); axes[1].set_ylabel('threshold(area)'); axes[1].legend()
            plt.tight_layout(); plt.show()